In [1]:
from steer_core.DataManager import DataManager

from steer_materials import (
    SeparatorMaterial, 
    InsulationMaterial, 
    CurrentCollectorMaterial,
    TapeMaterial, 
    AnodeMaterial,
    CathodeMaterial,
    ConductiveAdditive,
    Binder
)

# Simplified imports using __init__.py
from steer_opencell_design import (
    TablessCurrentCollector,
    AnodeFormulation,
    CathodeFormulation,
    Cathode, 
    Anode,
    Separator,
    Laminate,
    WoundJellyRoll,
    RoundMandrel,
    Tape,
    __version__
)

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.create_table(
    table_name='cells',
    columns={
        'name': 'TEXT PRIMARY KEY',
        'object': 'BLOB',
        'form_factor': 'TEXT',
        'internal_construction': 'TEXT',
        'date': 'TEXT',
        'version': 'TEXT',
        'reference': 'TEXT'
    }
)

In [4]:
db.remove_data(table_name='cells', condition="name = 'CBAK-32140NS'")

In [5]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'cathode_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials',
 'tape_materials']

In [6]:
# set some standard materials

conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation = InsulationMaterial.from_database("Aluminium Oxide, 95%")
current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
separator_material = SeparatorMaterial.from_database('Polyethylene')
tape_material = TapeMaterial.from_database("Kapton")


In [7]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=130,
    length=3200,
    coated_width=125,
    insulation_width=2.5,
    thickness=13.5
)

cathode_active_material = CathodeMaterial.from_database("NFM111 (Vendor C)")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=2.93,
    mass_loading=11.27,
    insulation_material=insulation,
    insulation_thickness=3
)

my_cathode.properties

{'Cost': '$ 1.17',
 'Mass': '103.7 g',
 'Coating mass': '88.36 g',
 'Total thickness': '90.4 um',
 'Coating thickness': '38.46 um'}

In [8]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=133,
    length=3250,
    coated_width=128,
    insulation_width=2.5,
    thickness=13.5,
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_material=insulation,
    insulation_thickness=3
)

my_anode.properties

{'Cost': '$ 0.67',
 'Mass': '81.19 g',
 'Coating mass': '65.26 g',
 'Total thickness': '159.0 um',
 'Coating thickness': '72.73 um'}

In [9]:
# create the layup 

top_separator = Separator(
    material=separator_material, 
    thickness=12,
    width = 130,
    length = 3600
)

bottom_serparator = Separator(
    material=separator_material, 
    thickness=12,
    width = 130,
    length = 3600,
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_serparator,
    name="CBAK-32140NS"
)



In [10]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)


tape = Tape(
    material = tape_material,
    thickness=30
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    tape=tape,
    additional_tape_wraps=3,
    name="CBAK-32140NS"
)


In [11]:
my_jellyroll.radius

17.27

In [12]:
my_jellyroll.roll_properties

,Turns
Anode A Side Coating Turns,52.50
Anode Current Collector Turns,52.50
Anode B Side Coating Turns,52.50
Cathode A Side Coating Turns,50.95
Cathode Current Collector Turns,50.95
Cathode B Side Coating Turns,50.95
Bottom Separator Turns,64.82
Bottom Separator Inner Turns,10.66
Bottom Separator Outer Turns,1.63
Top Separator Turns,64.82


In [13]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [14]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [15]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Cell 2,gASVVg8BAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:42:47,0.4.6,Na/Na+
1,Cell 4,gASVkQkBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:42:49,0.4.6,Na/Na+
2,HiNa-NaCR32140-MP10,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:42:52,0.4.6,Na/Na+
3,QNAS-S40160NL,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:42:54,0.4.6,Na/Na+
4,QNAS-S40160RL,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:42:56,0.4.6,Na/Na+
5,UniGrid-NCO32140,gASVPSEBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:42:59,0.4.6,Na/Na+
6,Vendor D NFPP,gASVGwABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-14 15:43:01,0.4.6,Na/Na+
7,Vendor E NFPP,gASVKQABAAAAAACMPnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-11-14 15:43:03,0.4.6,Na/Na+
8,Vendor G NFM,gASV9gwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:43:05,0.4.6,Na/Na+
9,Vendor G NFPP,gASV8iYBAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-14 15:43:07,0.4.6,Na/Na+
